# UFC Finish Dynamics (2015-2025)

## Executive summary

### research question / motivation
How has the UFC finish rate changed over time, how does it differ by weight class, and which fight-level features are most associated with fights ending inside the distance?

### data source
- scraped from [ufcstats.com](http://ufcstats.com/) event pages using `src/scraping.py`
- fight-level records include event metadata, bout stats, result labels, round/time, and bonus/title indicators
- this notebook uses the modern era subset from 2015-01-01 through 2025-12-31 and current UFC divisions

### key findings
- overall finish rate is relatively stable around 50%, with notable year-over-year swings (largest recent drop in 2024, rebound in 2025).
- heavier male divisions have the highest average finish rates, while women’s lower-weight divisions are more decision-heavy.
- finish-type mix differs by division: heavyweight outcomes skew toward KO/TKO, while women’s strawweight has the largest unanimous decision share.
- both logit and probit models show strong discrimination (`AUC ≈ 0.94`) and similar behavior, so threshold tuning matters more than model family.

### limitations / next steps
- this is observational, not causal; coefficient sign and magnitude should not be interpreted as causal effects.
- scraped stat fields can contain occasional missing or noisy values.
- add temporal validation and out-of-sample backtesting for deployment-grade prediction.
- enrich with fighter-level priors (age, stance, layoff, ranking context) for stronger inference.

## 1) setup

In [ ]:
from pathlib import Path
import sys
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.analysis_utils import (
    DEFAULT_HEATMAP_ORDER,
    DEFAULT_OUTCOME_ORDER,
    build_calibration_table,
    compute_auc_curves,
    compute_confusion_metrics,
    compute_division_finish_rates,
    compute_outcome_mix_by_division,
    compute_yearly_finish_stats,
    confusion_matrix_df,
    fit_finish_models,
    load_fight_data,
    prepare_analysis_frame,
    save_processed_data,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
sns.set_theme(style="whitegrid", context="talk")

figs_dir = project_root / "figs"
figs_dir.mkdir(parents=True, exist_ok=True)

## 2) load and prepare data

In [ ]:
raw_df = load_fight_data()
df = prepare_analysis_frame(raw_df, start_date="2015-01-01", end_date="2026-01-01")
processed_path = save_processed_data(df)

print(f"rows in analysis frame: {len(df):,}")
print(f"year span: {df['Year'].min()}-{df['Year'].max()}")
print(f"processed snapshot saved to: {processed_path}")

df.head()

## 3) how has the overall finish rate changed over time?

In [ ]:
yearly_stats = compute_yearly_finish_stats(df)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_stats.index, yearly_stats["Finish_Rate"], marker="o", linewidth=2.5)

for year, finish_rate, pct_change in zip(
    yearly_stats.index[1:],
    yearly_stats["Finish_Rate"].iloc[1:],
    yearly_stats["Finish_Rate_Change_Pct"].iloc[1:],
):
    if pd.notna(pct_change):
        color = "green" if pct_change > 0 else "red"
        ax.text(year, finish_rate + 0.015, f"{pct_change:+.1f}%", ha="center", fontsize=9, color=color)

ax.set_title("UFC Finish Rate Over Time (KO or Submission)", fontsize=16, weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("finish rate")
ax.set_ylim(0, 1)
ax.grid(alpha=0.35)
fig.tight_layout()
fig.savefig(figs_dir / "finish_rate_trend.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
largest_increase_year = int(yearly_stats["Finish_Rate_Change_Pct"].idxmax())
largest_decrease_year = int(yearly_stats["Finish_Rate_Change_Pct"].idxmin())
latest_year = int(yearly_stats.index.max())
latest_rate = yearly_stats.loc[latest_year, "Finish_Rate"]

summary = f"""
The aggregate finish rate remains centered around 50% across the window, with a low of {yearly_stats['Finish_Rate'].min():.3f} and a high of {yearly_stats['Finish_Rate'].max():.3f}.  
The sharpest increase occurred in **{largest_increase_year}** ({yearly_stats.loc[largest_increase_year, 'Finish_Rate_Change_Pct']:+.1f}% vs prior year), while the sharpest decline occurred in **{largest_decrease_year}** ({yearly_stats.loc[largest_decrease_year, 'Finish_Rate_Change_Pct']:+.1f}%).  
In **{latest_year}**, the finish rate is **{latest_rate:.3f}**, which supports the view that yearly volatility exists but there is no persistent drift away from the ~0.5 baseline.
"""
display(Markdown(summary))

## 4) which divisions finish most often?

In [ ]:
heatmap_data = compute_division_finish_rates(df, weight_order=DEFAULT_HEATMAP_ORDER)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    heatmap_data,
    cmap="Blues",
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    linecolor="gray",
    cbar_kws={"label": "finish rate"},
    ax=ax,
)
ax.set_title("Finish Rate by Division and Year", fontsize=16, weight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Weight class")
fig.tight_layout()
fig.savefig(figs_dir / "division_finish_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
division_mean_rates = heatmap_data.mean(axis=1).dropna().sort_values(ascending=False)
top_division = division_mean_rates.index[0]
bottom_division = division_mean_rates.index[-1]

summary = f"""
Average finish rates rank **{top_division}** highest ({division_mean_rates.iloc[0]:.3f}) and **{bottom_division}** lowest ({division_mean_rates.iloc[-1]:.3f}).  
The heatmap pattern indicates a durable gradient: heavier classes are generally more finish-prone than lighter classes, with women’s divisions tending toward lower finish frequencies.
"""
display(Markdown(summary))

## 5) how do finish types differ across divisions?

In [ ]:
outcome_mix = compute_outcome_mix_by_division(df, weight_order=DEFAULT_OUTCOME_ORDER)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(outcome_mix, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title("Outcome Mix by Division", fontsize=16, weight="bold")
ax.set_xlabel("Outcome")
ax.set_ylabel("Weight class")
fig.tight_layout()
fig.savefig(figs_dir / "division_outcome_mix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
heavyweight_ko = outcome_mix.loc["Heavyweight", "KO"]
heavyweight_udec = outcome_mix.loc["Heavyweight", "UDEC"]
straw_ko = outcome_mix.loc["Women's Strawweight", "KO"]
straw_udec = outcome_mix.loc["Women's Strawweight", "UDEC"]

summary = f"""
**Heavyweight** bouts are KO-dominant (KO share: {heavyweight_ko:.3f}) and less decision-heavy (UDEC share: {heavyweight_udec:.3f}).  
**Women's Strawweight** shows the opposite profile (KO share: {straw_ko:.3f}, UDEC share: {straw_udec:.3f}).  
This supports division-specific tactical and physical dynamics rather than a one-size-fits-all finish process.
"""
display(Markdown(summary))

## 6) predictive modeling: logit vs probit for finish probability

In [ ]:
bundle = fit_finish_models(df)
auc_payload = compute_auc_curves(bundle)

model_summary = pd.DataFrame(
    {
        "metric": ["observations", "logit pseudo r2", "probit pseudo r2", "logit auc", "probit auc"],
        "value": [
            len(bundle.y),
            bundle.logit_model.prsquared,
            1 - (bundle.probit_model.llf / bundle.probit_model.llnull),
            auc_payload["auc_logit"],
            auc_payload["auc_probit"],
        ],
    }
)
model_summary

In [ ]:
key_terms = [
    "Round",
    "Title",
    "Total_KD",
    "Total_SIG_STR",
    "Total_TD",
    "Total_SUB_ATT",
    "SIG_STR_DIFF_ABS",
]

coef_table = pd.DataFrame(
    {
        "coef": bundle.logit_model.params[key_terms],
        "p_value": bundle.logit_model.pvalues[key_terms],
    }
)
coef_table["odds_ratio"] = np.exp(coef_table["coef"])
coef_table.sort_values("p_value")

In [ ]:
calibration = build_calibration_table(bundle)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(calibration["Logit_Pred"], calibration["Observed_Finish"], marker="o", linewidth=2, label="Logit")
ax.plot(calibration["Probit_Pred"], calibration["Observed_Finish"], marker="s", linewidth=2, label="Probit")
ax.plot([0, 1], [0, 1], "k--", alpha=0.7, label="Perfect calibration")
ax.set_title("Calibration: Predicted vs Observed Finish Rate", fontsize=15, weight="bold")
ax.set_xlabel("mean predicted finish probability (bin)")
ax.set_ylabel("observed finish rate (bin)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(figs_dir / "model_calibration.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
calibration = calibration.copy()
calibration["logit_error"] = calibration["Observed_Finish"] - calibration["Logit_Pred"]

summary = f"""
Calibration tracks the diagonal closely across most bins, indicating that estimated probabilities are reasonably well calibrated.  
Largest underprediction from logit is about **{calibration['logit_error'].max():.3f}**, while largest overprediction is about **{calibration['logit_error'].min():.3f}**.  
Overall differences between logit and probit calibration are small for practical decision-making.
"""
display(Markdown(summary))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(auc_payload["fpr_logit"], auc_payload["tpr_logit"], linewidth=2, label=f"Logit (AUC={auc_payload['auc_logit']:.3f})")
ax.plot(auc_payload["fpr_probit"], auc_payload["tpr_probit"], linewidth=2, label=f"Probit (AUC={auc_payload['auc_probit']:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.7, label="Random baseline")
ax.set_title("ROC Curve: Logit vs Probit", fontsize=15, weight="bold")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(figs_dir / "roc_logit_probit.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
cm_logit = confusion_matrix_df(auc_payload["y_true"], bundle.logit_prob, threshold=0.5)
cm_probit = confusion_matrix_df(auc_payload["y_true"], bundle.probit_prob, threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm_logit, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("logit confusion matrix (threshold=0.5)")
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("actual")

sns.heatmap(cm_probit, annot=True, fmt="d", cmap="Greens", cbar=False, ax=axes[1])
axes[1].set_title("probit confusion matrix (threshold=0.5)")
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("actual")

fig.tight_layout()
fig.savefig(figs_dir / "confusion_logit_probit.png", dpi=200, bbox_inches="tight")
plt.show()

metrics_table = pd.DataFrame(
    {
        "logit": compute_confusion_metrics(auc_payload["y_true"], bundle.logit_prob, threshold=0.5),
        "probit": compute_confusion_metrics(auc_payload["y_true"], bundle.probit_prob, threshold=0.5),
    }
).T
metrics_table

In [ ]:
logit_metrics = compute_confusion_metrics(auc_payload["y_true"], bundle.logit_prob, threshold=0.5)
probit_metrics = compute_confusion_metrics(auc_payload["y_true"], bundle.probit_prob, threshold=0.5)

summary = f"""
Both models discriminate strongly (**logit AUC={auc_payload['auc_logit']:.4f}**, **probit AUC={auc_payload['auc_probit']:.4f}**), and their confusion profiles are nearly identical at a 0.5 threshold.  
Logit has slightly higher precision ({logit_metrics['precision']:.3f}) and specificity ({logit_metrics['specificity']:.3f}), while probit has marginally higher recall ({probit_metrics['recall']:.3f}).  
Given this near-parity, threshold selection and business objective (false alarms vs missed finishes) should drive operational tuning more than link-function choice.
"""
display(Markdown(summary))

## 7) decision takeaway

The finish ecosystem is structurally segmented by division and only moderately volatile year to year. For tactical use (matchmaking, betting models, or content planning), a probability model in this feature family is reliable for ranking bouts by finish likelihood, and post-model threshold calibration is the highest-leverage next step.